# VideoDB Understanding: Activity

Turn a video into a timeline of actions and goals with the `activity` analyzer. Use VideoDB or an Open Compute model, then inspect the actions and play the corresponding video moments.

This example uses a first-person video of a person folding laundry. Activity handles frame preparation and both analysis passes for you.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/indexing-v2/understanding/activity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install dependencies

In [ ]:
%pip install -q "videodb>=0.5.0" python-dotenv pandas

## 2. Connect to VideoDB

Set `VIDEO_DB_BASE_URL` in your environment or `.env` to your VideoDB endpoint with Activity support deployed. Use your VideoDB API key when prompted.

The VideoDB example uses the server's default Activity model and requires a deployment that supports omitting the model. For Open Compute, reuse or create a sandbox below with a model that supports both vision and text generation.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
api_key = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")

conn = connect(api_key=api_key, base_url=os.environ["VIDEO_DB_BASE_URL"])
collection = conn.get_collection()

## 3. Choose a video

Upload the sample video directly from its YouTube URL. Replace it with your own URL, or reuse an uploaded video.

For the optional local download, uncomment the yt-dlp cell, then use the commented file upload instead of the URL upload. Merging video and audio requires `ffmpeg`, which Colab already has.

In [ ]:
# Optional: download locally instead of uploading the URL.
# %pip install -q yt-dlp
# !yt-dlp -q -f "bv*[ext=mp4][height<=1080]+ba[ext=m4a]/b[ext=mp4]/b" --merge-output-format mp4 -o laundry-folding-pov.mp4 "https://www.youtube.com/watch?v=wuRf25_zME0"

In [ ]:
video = collection.upload(url="https://www.youtube.com/watch?v=wuRf25_zME0")

# Alternatively, after running the optional download cell, replace the URL upload with:
# video = collection.upload(file_path="laundry-folding-pov.mp4")

# Or reuse an uploaded video:
# video = collection.get_video("YOUR_VIDEO_ID")

video.play()

## 4. VideoDB: run and view results

This section includes submission, waiting, results and playback. Run this section for VideoDB, or skip directly to **Open Compute** below. Each submission creates a separate Understanding run.

- **`express`** starts processing immediately; selected in this example.
- **`batch`** offers 50% savings on model inference costs, but can take longer.

Both modes return an Understanding ID immediately; wait for the results below. VideoDB selects the default Activity model.


In [ ]:
MODE = "express"  # Use "batch" for 50% savings on model inference costs.

activity_run = video.understand(analyzers=[{
    "type": "activity",
    "name": "activity",
    "config": {"mode": MODE},
}])
print("Understanding ID:", activity_run.id)

### Wait for Activity results

Batch jobs can take longer than express requests. If this cell times out, rerun **this cell** to keep waiting for the same Understanding ID. Rerunning a submission cell creates another run.

In [ ]:
activity_run.wait_until_complete(timeout=48 * 3600, poll_interval=60)
activity_analyzer = activity_run.get_analyzer("activity", refresh=True)
print("Status:", activity_analyzer.status)
if not activity_analyzer.is_successful:
    raise RuntimeError(f"Activity did not succeed: {activity_analyzer.status}")

activity_output = activity_analyzer.get_output()

### Read the action timeline

`scenes` contains the actions. Each entry has `scene_id`, `start` and `end` in seconds, plus `data` describing the action, its goal, confidence and any issue. `metadata.goals` groups actions into higher-level goals.

Warnings matter: a completed run may contain uncertain or unavailable intervals. Review warnings and rows with an issue before relying on the timeline.

Results contain activity annotations and processing warnings. Billing details are not part of the analyzer output.

In [ ]:
import pandas as pd
from IPython.display import display

activity_scenes = activity_output.get("scenes", [])
activity_metadata = activity_output.get("metadata", {})
if activity_metadata.get("warnings"):
    print("Processing warnings:", activity_metadata["warnings"])

activity_rows = []
for activity_scene in activity_scenes:
    activity_data = activity_scene.get("data") or {}
    activity_rows.append({
        "Scene ID": activity_scene["scene_id"],
        "Start (s)": activity_scene["start"],
        "End (s)": activity_scene["end"],
        "Action": activity_data.get("instruction"),
        "Goal": activity_data.get("goal_instruction"),
        "Confidence": activity_data.get("confidence"),
        "Issue": activity_data.get("issue"),
    })
activity_actions = pd.DataFrame(activity_rows)
display(activity_actions.head(20))

### Goals

A goal can span several actions. An unavailable start or end means the output did not contain enough supporting actions to establish that goal's interval.

In [ ]:
activity_goals = pd.DataFrame(activity_metadata.get("goals", []))
activity_goal_columns = [column for column in ["goal_id", "instruction", "start", "end", "uncertainty"] if column in activity_goals]
display(activity_goals[activity_goal_columns])

### Play an action

Choose a row from the action table to generate a stream for just that moment. The stream URL can also be used in your own player. Generating a stream does not rerun Activity.

In [ ]:
from videodb import play_stream

ACTION_INDEX = 0  # Change this to another row number.
if not activity_scenes:
    raise ValueError("No action intervals were returned.")
activity_scene = activity_scenes[ACTION_INDEX]
start, end = activity_scene.get("start"), activity_scene.get("end")
if start is None or end is None or end <= start:
    raise ValueError("Choose an action with a valid time interval.")

activity_stream_url = video.generate_stream(timeline=[(start, end)])
print(f"{activity_scene['scene_id']}: {start:.2f}s–{end:.2f}s")
print("Stream URL:", activity_stream_url)
play_stream(activity_stream_url)

## 5. Open Compute: run and view results

This is a separate Open Compute example. Skip it if you only want results from the VideoDB section. You can also run it directly after the shared setup, without running the VideoDB section.

This example uses `Qwen/Qwen3.5-9B`. Open Compute handles inference as jobs; **omit `mode` entirely**. Sending `mode`, including `batch`, is rejected.

Supply a compatible sandbox ID below to reuse it, or leave it empty to create a small sandbox for the selected model. The model must support both vision and text generation. The created sandbox is named `Activity cookbook`.


In [ ]:
from videodb import SandboxTier

OPEN_COMPUTE_MODEL = "Qwen/Qwen3.5-9B"
SANDBOX_ID = ""  # Optional: a compatible sandbox already in your account.


In [ ]:
if SANDBOX_ID:
    sandbox = conn.get_sandbox(SANDBOX_ID)
else:
    sandbox = conn.create_sandbox(
        tier=SandboxTier.small,
        models=[OPEN_COMPUTE_MODEL],
        name="Activity cookbook",
    )
    SANDBOX_ID = sandbox.id
    created_sandbox_id = sandbox.id

print("Sandbox ID:", sandbox.id)


Wait for the sandbox to become ready before submitting Activity. If the wait times out, rerun only the next cell to keep waiting for the same sandbox.


In [ ]:
sandbox.wait_for_ready(timeout=600, interval=15)
print("Sandbox status:", sandbox.status)


Submit Activity using the sandbox selected above.


In [ ]:
activity_config = {"model": OPEN_COMPUTE_MODEL}
if SANDBOX_ID:
    activity_config["sandbox_id"] = SANDBOX_ID

open_compute_run = video.understand(analyzers=[{
    "type": "activity",
    "name": "activity",
    "config": activity_config,
}])
print("Understanding ID:", open_compute_run.id)

### Wait for Open Compute results

If this cell times out, rerun **this cell** to keep waiting for the same Understanding ID. Rerunning a submission cell creates another run.

In [ ]:
open_compute_run.wait_until_complete(timeout=48 * 3600, poll_interval=60)
open_compute_analyzer = open_compute_run.get_analyzer("activity", refresh=True)
print("Status:", open_compute_analyzer.status)
if not open_compute_analyzer.is_successful:
    raise RuntimeError(f"Activity did not succeed: {open_compute_analyzer.status}")

open_compute_activity_output = open_compute_analyzer.get_output()

### Read the action timeline

`scenes` contains the actions. Each entry has `scene_id`, `start` and `end` in seconds, plus `data` describing the action, its goal, confidence and any issue. `metadata.goals` groups actions into higher-level goals.

Warnings matter: a completed run may contain uncertain or unavailable intervals. Review warnings and rows with an issue before relying on the timeline.

Results contain activity annotations and processing warnings. Billing details are not part of the analyzer output.

In [ ]:
import pandas as pd
from IPython.display import display

open_compute_scenes = open_compute_activity_output.get("scenes", [])
open_compute_metadata = open_compute_activity_output.get("metadata", {})
if open_compute_metadata.get("warnings"):
    print("Processing warnings:", open_compute_metadata["warnings"])

open_compute_rows = []
for open_compute_scene in open_compute_scenes:
    open_compute_data = open_compute_scene.get("data") or {}
    open_compute_rows.append({
        "Scene ID": open_compute_scene["scene_id"],
        "Start (s)": open_compute_scene["start"],
        "End (s)": open_compute_scene["end"],
        "Action": open_compute_data.get("instruction"),
        "Goal": open_compute_data.get("goal_instruction"),
        "Confidence": open_compute_data.get("confidence"),
        "Issue": open_compute_data.get("issue"),
    })
open_compute_actions = pd.DataFrame(open_compute_rows)
display(open_compute_actions.head(20))

### Goals

A goal can span several actions. An unavailable start or end means the output did not contain enough supporting actions to establish that goal's interval.

In [ ]:
open_compute_goals = pd.DataFrame(open_compute_metadata.get("goals", []))
open_compute_goal_columns = [column for column in ["goal_id", "instruction", "start", "end", "uncertainty"] if column in open_compute_goals]
display(open_compute_goals[open_compute_goal_columns])

### Play an action

Choose a row from the action table to generate a stream for just that moment. The stream URL can also be used in your own player. Generating a stream does not rerun Activity.

In [ ]:
from videodb import play_stream

ACTION_INDEX = 0  # Change this to another row number.
if not open_compute_scenes:
    raise ValueError("No action intervals were returned.")
open_compute_scene = open_compute_scenes[ACTION_INDEX]
start, end = open_compute_scene.get("start"), open_compute_scene.get("end")
if start is None or end is None or end <= start:
    raise ValueError("Choose an action with a valid time interval.")

open_compute_stream_url = video.generate_stream(timeline=[(start, end)])
print(f"{open_compute_scene['scene_id']}: {start:.2f}s–{end:.2f}s")
print("Stream URL:", open_compute_stream_url)
play_stream(open_compute_stream_url)

### Clean up the sandbox (optional)

After your Activity run finishes and you are done with the sandbox, run this cell to stop the sandbox created by this notebook. A supplied sandbox is left running. Skip this cell if you want to keep using the created sandbox.


In [ ]:
if globals().get("created_sandbox_id"):
    owned_sandbox = conn.get_sandbox(created_sandbox_id)
    owned_sandbox.stop(grace=True)
    owned_sandbox.wait_for_stop(timeout=300, interval=15)
    print("Sandbox status:", owned_sandbox.status)
else:
    print("No sandbox created by this notebook; nothing to stop.")
